## Preparación opcional: instalar las bibliotecas

para la instalacion de pandas y matplotlib

In [ ]:
%pip install pandas matplotlib


## 1. Importar Pandas y localizar la carpeta del proyecto

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if "data" not in os.listdir(ROOT):
    os.chdir("..")
    ROOT = os.getcwd()

RAW = os.path.join(ROOT, "data", "raw")
INPUT = os.path.join(ROOT, "data", "input")
PROCESSED = os.path.join(ROOT, "data", "processed")
CHARTS = os.path.join(ROOT, "outputs", "graficos")


## 2. Cargar y explorar las campañas

Comprueb0 las primeras filas, el tamaño, los nombres de columnas y los tipos de datos.

In [ ]:
campaigns_raw = pd.read_csv(os.path.join(RAW, "bank-additional.csv"))

print("Tamaño:", campaigns_raw.shape)
campaigns_raw.head()


In [ ]:
campaigns_raw.info()


## 3. Revisar valores ausentes y duplicados

Esta revisión me ayuda a decidir qué limpiar. En este análisis los valores ausentes los conservo, no se rellenan.

In [ ]:
campaigns_raw.isnull().sum()


In [ ]:
print("Filas duplicadas:", campaigns_raw.duplicated().sum())
print("Identificadores id_ duplicados:", campaigns_raw["id_"].duplicated().sum())


## 4. Limpiar las campañas

quito la columna de índice exportada y convierto las cuatro columnas con coma decimal para poder calcular estadísticas.

In [ ]:
campaigns = campaigns_raw

unnamed_columns = []
for column in campaigns.columns:
    if "Unnamed:" in column:
        unnamed_columns.append(column)

campaigns = campaigns.drop(unnamed_columns, axis=1)

decimal_columns = ["cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]
for column in decimal_columns:
    campaigns[column] = campaigns[column].str.replace(",", ".").astype(float)

print("Tamaño después de quitar el índice:", campaigns.shape)
campaigns.head()


## 5. Cargar las fichas de clientes

El CSV de entrada contiene las tres hojas del Excel apiladas y conserva el año de procedencia. Así el notebook lee ambos conjuntos con read csv

In [ ]:
customers = pd.read_csv(os.path.join(INPUT, "customer-details-combinado.csv"))

print("Fichas de clientes:", customers.shape)
customers.head()


## 6. Combinar campañas y clientes

Se mantienen todas las campañas y compruebo cuántas encontraron una ficha.

In [ ]:
merged = campaigns.merge(
    customers, how="left", left_on="id_", right_on="ID"
)
combined = merged.drop(["ID"], axis=1)

print("Campañas:", len(campaigns))
print("Fichas combinadas:", len(customers))
print("Campañas con ficha enlazada:", combined["Income"].count())
print("Tamaño del cruce:", combined.shape)


## 7. Analizar el resultado general

"y" indica si hubo suscripción. Reviso los conteos y calculo la tasa sobre el total.

In [ ]:
target_counts = campaigns["y"].value_counts()
target_counts


In [ ]:
subscription_rate = target_counts["yes"] / len(campaigns)
print("Tasa de suscripción:", f"{subscription_rate:.2%}")


## 8. Comparar la tasa por canal

Agrupo los registros por canal y cuento los contactos y las suscripciones.

In [ ]:
channel_summary = campaigns.groupby("contact")["y"].agg(
    ["count", lambda values: (values == "yes").sum()]
)
channel_summary.columns = ["contacts", "subscriptions"]
channel_summary["rate"] = channel_summary["subscriptions"] / channel_summary["contacts"]
channel_summary


## 9. Comparar según el resultado de la campaña anterior

In [ ]:
previous_summary = campaigns.groupby("poutcome")["y"].agg(
    ["count", lambda values: (values == "yes").sum()]
)
previous_summary.columns = ["contacts", "subscriptions"]
previous_summary["rate"] = previous_summary["subscriptions"] / previous_summary["contacts"]
previous_summary


## 10. Revisar duración, ingresos y visitas web

Resymen descriptivo por resultado.

In [ ]:
duration_summary = combined.groupby("y")["duration"].agg(["mean", "median"])
duration_summary


In [ ]:
income_summary = combined.groupby("y")["Income"].agg(["mean", "median"])
income_summary


In [ ]:
visits_summary = combined.groupby("y")["NumWebVisitsMonth"].agg(["mean", "median"])
visits_summary


## 11. Crear gráficos con Matplotlib

Creo los graficos para verlos en notebook y se guardan como imágenes PNG en outputs/graficos.

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
target_summary = pd.DataFrame({
    "contacts": target_counts,
    "rate": target_counts / len(campaigns),
})
target_summary


### 11.1 Resultado de las campañas

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(target_counts.index, target_counts.values, color=["#7295b5", "#4c9f70"])
plt.title("Resultado de las campañas")
plt.xlabel("Suscripción")
plt.ylabel("Número de campañas")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "01_resultado_campana.png"))
plt.show()


### 11.2 Tasa de suscripción por canal

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(channel_summary.index, channel_summary["rate"] * 100, color="#397d78")
plt.title("Tasa de suscripción por canal")
plt.xlabel("Canal")
plt.ylabel("Tasa de suscripción (%)")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "02_tasa_por_canal.png"))
plt.show()


### 11.3 Tasa según el resultado anterior

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(previous_summary.index, previous_summary["rate"] * 100, color="#d47a43")
plt.title("Tasa según el resultado de la campaña anterior")
plt.xlabel("Resultado anterior")
plt.ylabel("Tasa de suscripción (%)")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "03_tasa_resultado_previo.png"))
plt.show()


### 11.4 Mediana de duración según la suscripción

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(duration_summary.index, duration_summary["median"], color="#8c78a8")
plt.title("Mediana de duración de llamada según resultado")
plt.xlabel("Suscripción")
plt.ylabel("Duración (segundos)")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "04_mediana_duracion.png"))
plt.show()


In [ ]:
channel_summary


In [ ]:
previous_summary


## 12. Guardar los datos preparados

Conserva los archivos originales y guarda estas versiones para consultar el resultado del análisis.

In [ ]:
campaigns.to_csv(os.path.join(PROCESSED, "bank-additional-limpio.csv"), index=False)
customers.to_csv(os.path.join(PROCESSED, "customer-details-combinado.csv"), index=False)
combined.to_csv(os.path.join(PROCESSED, "campanas-clientes-combinado.csv"), index=False)

print("Tablas guardadas en:", PROCESSED)


## 13. Conclusiones



De las 43.000 campañas analizadas, 4.844 terminaron en suscripción, lo que representa una tasa general del 11,27 %. La tasa fue mayor en el canal cellular, con un 14,74 % (4.039 de 27.396), que en telephone, con un 5,16 % (805 de 15.604).

El resultado de la campaña anterior también muestra una diferencia: se suscribió el 65,32 % de los clientes cuyo resultado previo fue SUCCESS (938 de 1.436). La tasa fue del 14,23 % para FAILURE y del 8,82 % para NONEXISTENT. El grupo SUCCESS es más pequeño, por lo que conviene interpretar su porcentaje con cautela.

La mediana de duración de las llamadas fue de 449 segundos entre quienes se suscribieron y de 163 segundos entre quienes no lo hicieron. Como la duración solo se conoce al terminar la llamada, esta diferencia describe lo ocurrido, pero no sirve para decidir a quién llamar antes de la interacción.

Los ingresos y las visitas web mensuales muestran medias parecidas entre los grupos con suscripción y sin ella. En conjunto, los datos muestran diferencias por canal y por resultado anterior, pero este análisis descriptivo no permite afirmar que esas variables causen la suscripción.

Limitaciones y decisiones de preparación

La columna date no se convirtió ni se utilizó para analizar tendencias temporales. Los valores ausentes se conservaron sin imputar. El Excel original se mantiene en data/raw/; el notebook utiliza el CSV preparado de las tres hojas, guardado en data/input/.